# Close Faster, Audit Smarter: Build It with CoCo, Ask It with CoWork

**Meridian Stay** is a global hotel brand. Every month-end close, the finance team manually reconciles three systems — payroll, general ledger, and the bank — hunting for the one transaction that doesn't tie out. Last quarter, a $47,000 wire transfer was never journaled in the GL and the auditors flagged it.

In this lab you'll fix that:

1. **Land** the raw financial exports in open **Apache Iceberg** tables.
2. **Clean and standardize** them with **CoCo** into a unified transaction ledger.
3. **Match** transactions across systems and **flag every discrepancy**.
4. **See** the reconciliation status on a live **Streamlit** dashboard.
5. **Ask** close-cycle questions in plain language with **Snowflake CoWork** (done in the Snowsight UI — see the guide).

Some cells are pre-written; others you'll generate by prompting **CoCo** (the prompt is in the markdown cell right above each one).

**Schemas:** `RAW` (exports as they arrived) → `CURATED` (clean, governed tables) → `ANALYTICS` (semantic view and agent).

## Connect this notebook to compute

This notebook runs inside a **Git-backed Workspace**. Before running any cell:

1. Click **Connect** next to the Run button and select **Create and connect**. Wait for the status bar at the bottom of the notebook to show **Connected**.

2. Set your **role** and **warehouse** using the picker at the top of the Notebooks editor:
   - **Role = `ACCOUNTADMIN`**
   - **Warehouse = `COMPUTE_WH`** (the default trial warehouse)

3. Notebooks in Workspaces **don't auto-select a database or schema**, so every cell below uses fully qualified names (`MERIDIAN_STAY_FINANCE.RAW.…`).

4. Run cells top-to-bottom with **▶** / **Shift+Enter**. If the session idle-suspends, restart it and re-run from where you left off — objects you created persist in Snowflake.

Where a markdown cell shows a **CoCo prompt**, open the **CoCo** panel, paste the prompt, compare CoCo's output against the **Expected output** in the cell, and click **Allow** to run it. Optionally, copy the SQL into the empty cell below the prompt for future reference.

In [ ]:
-- SETUP: create the Meridian Stay Finance database and grant the Cortex privileges used later in CoWork.
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE COMPUTE_WH;

CREATE OR REPLACE DATABASE MERIDIAN_STAY_FINANCE;
CREATE OR REPLACE SCHEMA MERIDIAN_STAY_FINANCE.RAW;
CREATE OR REPLACE SCHEMA MERIDIAN_STAY_FINANCE.CURATED;
CREATE OR REPLACE SCHEMA MERIDIAN_STAY_FINANCE.ANALYTICS;

-- Privileges needed later to create the Cortex Agent and use CoWork
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_AGENT_USER TO ROLE ACCOUNTADMIN;
ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'ANY_REGION';

## Land the raw exports in Apache Iceberg tables

The three financial exports live in this repo's `data/` folder:

| File | Source system | How it's messy |
|---|---|---|
| `payroll_register.csv` | Payroll (ADP / Workday) | `MM/DD/YYYY` dates, amounts as text (`"$4,200.00"`), department codes like `F&B` and `MGMT`, and one row exported twice |
| `gl_journal_entries.csv` | General ledger (NetSuite / SAP) | `Nov 01 2026` dates, debits in parentheses (`(23,900.00)`), category codes (`PAY`, `UTIL`, `MAINT`), and a **Reversed** entry that was never removed |
| `bank_statement.csv` | Bank | `YYYY-MM-DD` dates, debits as negative numbers, free-text descriptions (`"WIRE TFR - GRAND BALLROOM VENUE DEPOSIT"`), and no category or department columns |

We land each one, exactly as it arrived, in an **Apache Iceberg** table. Iceberg is an open table format: the data is stored as Parquet files plus open metadata, so other engines (Spark, Trino, DuckDB, …) can read the same tables through Snowflake Horizon Catalog.

`CATALOG = 'SNOWFLAKE'` + `EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED'` means Snowflake manages the Iceberg storage for you — no cloud bucket or IAM setup needed.

Run the next cell as-is. It creates the three raw Iceberg tables (every column as `STRING`, exactly as exported), a CSV file format, and an internal stage for the files.

In [ ]:
CREATE OR REPLACE ICEBERG TABLE MERIDIAN_STAY_FINANCE.RAW.PAYROLL_REGISTER (
    employee_id    STRING,
    employee_name  STRING,
    department     STRING,
    pay_date       STRING,
    pay_type       STRING,
    gross_amount   STRING,
    net_amount     STRING,
    reference_id   STRING
)
  CATALOG = 'SNOWFLAKE'
  EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED';

CREATE OR REPLACE ICEBERG TABLE MERIDIAN_STAY_FINANCE.RAW.GL_JOURNAL_ENTRIES (
    journal_id    STRING,
    posting_date  STRING,
    account_code  STRING,
    account_name  STRING,
    description   STRING,
    amount        STRING,
    department    STRING,
    category      STRING,
    reference_id  STRING,
    status        STRING
)
  CATALOG = 'SNOWFLAKE'
  EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED';

CREATE OR REPLACE ICEBERG TABLE MERIDIAN_STAY_FINANCE.RAW.BANK_STATEMENT (
    transaction_id    STRING,
    transaction_date  STRING,
    description       STRING,
    amount            STRING,
    type              STRING,
    balance           STRING
)
  CATALOG = 'SNOWFLAKE'
  EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED';

CREATE OR REPLACE FILE FORMAT MERIDIAN_STAY_FINANCE.RAW.CSV_FF
  TYPE = 'CSV'
  SKIP_HEADER = 1
  FIELD_OPTIONALLY_ENCLOSED_BY = '"';

CREATE OR REPLACE STAGE MERIDIAN_STAY_FINANCE.RAW.FINANCIAL_EXPORTS
  FILE_FORMAT = MERIDIAN_STAY_FINANCE.RAW.CSV_FF;

### Upload the export files to the stage

This Python cell copies the three CSVs from the repo's `data/` folder into the `FINANCIAL_EXPORTS` stage. Run it as-is — you should see three files with status `UPLOADED`.

In [ ]:
import os
from snowflake.snowpark.context import get_active_session

session = get_active_session()

for f in ["payroll_register.csv", "gl_journal_entries.csv", "bank_statement.csv"]:
    result = session.file.put(
        os.path.abspath(os.path.join("data", f)),
        "@MERIDIAN_STAY_FINANCE.RAW.FINANCIAL_EXPORTS",
        auto_compress=False,
        overwrite=True,
    )
    print(result[0].source, result[0].status)

### STEP 1 — Load the exports into the raw Iceberg tables

`COPY INTO` is Snowflake's bulk-loading command — it reads files from a stage and inserts them into a table. It works the same way whether the target is a standard table or an Iceberg table.

Open the **CoCo** panel and send this prompt. Compare CoCo's output against the expected output below — if they match, click **Allow** to run it.

> *"Load the three CSV files in the stage @MERIDIAN_STAY_FINANCE.RAW.FINANCIAL_EXPORTS into their matching Iceberg tables in MERIDIAN_STAY_FINANCE.RAW: payroll_register.csv into PAYROLL_REGISTER, gl_journal_entries.csv into GL_JOURNAL_ENTRIES, and bank_statement.csv into BANK_STATEMENT. Use the file format MERIDIAN_STAY_FINANCE.RAW.CSV_FF and load the columns in file order."*

You should see **13**, **18**, and **18** rows loaded. If CoCo's first attempt fails and it retries with a corrected statement, that's normal; it reads the error and fixes its own SQL.

CoCo may add `MATCH_BY_COLUMN_NAME = NONE` to each statement. That's fine: it loads columns by position, which is the default and exactly what *"in file order"* asks for.

<details>
<summary>Expected output (check against this)</summary>

```sql
COPY INTO MERIDIAN_STAY_FINANCE.RAW.PAYROLL_REGISTER
  FROM @MERIDIAN_STAY_FINANCE.RAW.FINANCIAL_EXPORTS/payroll_register.csv
  FILE_FORMAT = (FORMAT_NAME = MERIDIAN_STAY_FINANCE.RAW.CSV_FF);

COPY INTO MERIDIAN_STAY_FINANCE.RAW.GL_JOURNAL_ENTRIES
  FROM @MERIDIAN_STAY_FINANCE.RAW.FINANCIAL_EXPORTS/gl_journal_entries.csv
  FILE_FORMAT = (FORMAT_NAME = MERIDIAN_STAY_FINANCE.RAW.CSV_FF);

COPY INTO MERIDIAN_STAY_FINANCE.RAW.BANK_STATEMENT
  FROM @MERIDIAN_STAY_FINANCE.RAW.FINANCIAL_EXPORTS/bank_statement.csv
  FILE_FORMAT = (FORMAT_NAME = MERIDIAN_STAY_FINANCE.RAW.CSV_FF);
```
</details>

### See the mess

Before cleaning anything, look at what arrived. This query shows rows from each of the three exports. Run it as-is and scan the results.

Things to notice:

- **Payroll:** James Rivera (EMP-102) appears **twice** on 11/15 with the same reference ID — a re-export duplicate. Amounts are text like `"$4,200.00"`. Departments say `F&B` and `MGMT` instead of full names.
- **GL:** Amounts are in parentheses for debits: `(23,900.00)`. Dates say `Nov 01 2026`. JE-2014 is a **Reversed** entry that cancels JE-2013, but both are still in the export. Categories are codes: `PAY`, `UTIL`, `MAINT`.
- **Bank:** The cleanest of the three — ISO dates, negative numbers for debits — but the descriptions are free text (`"WIRE TFR - GRAND BALLROOM VENUE DEPOSIT"`), and there's no category or department at all.

In [ ]:
SELECT 'PAYROLL_REGISTER' AS source, reference_id AS id, employee_name AS description,
       department, pay_date AS date, gross_amount AS amount, NULL AS status
FROM MERIDIAN_STAY_FINANCE.RAW.PAYROLL_REGISTER
UNION ALL
SELECT 'GL_JOURNAL_ENTRIES', journal_id, description, department, posting_date, amount, status
FROM MERIDIAN_STAY_FINANCE.RAW.GL_JOURNAL_ENTRIES
UNION ALL
SELECT 'BANK_STATEMENT', transaction_id, description, NULL, transaction_date, amount, NULL
FROM MERIDIAN_STAY_FINANCE.RAW.BANK_STATEMENT
ORDER BY source, date;

## Clean and standardize with CoCo

Now you'll turn three messy exports into one unified transaction ledger. This usually takes a finance analyst a full day of spreadsheet work. With CoCo, you describe the rules in plain language and review the SQL it writes.

The rules:

- **One vocabulary.** Every department and expense category maps to a single standard label.
- **Real dates and numbers.** Three date formats become real `DATE` values. Text amounts (`"$4,200.00"`), parenthesized debits (`(23,900.00)`), and negative numbers (`-23900.00`) all become clean `NUMBER(12,2)` values.
- **No duplicates.** A payroll row that was exported twice is kept once.
- **No reversed entries.** A reversed GL entry and the original entry it cancels net to zero, so neither belongs in the ledger.

### STEP 2 — Let CoCo find the mess

Before building anything, ask CoCo to compare the three sources. It will query the raw tables itself and show you how differently each system records the *same* things. The standard values in the prompt are your team's chart of accounts, so every system ends up speaking the same language.

> *"Compare how the three tables in MERIDIAN_STAY_FINANCE.RAW record departments and expense categories. Map every label to one of these standard values:*
> - *Department: Front Desk, Food & Beverage, Housekeeping, Management, Spa & Wellness, Operations, Facilities, Marketing, Admin, Finance*
> - *Category: Payroll, Utilities, Maintenance, Marketing, Travel, Office Supplies, Insurance, Professional Services*
>
> *Also point out how dates and amounts are formatted differently, and any rows that should be excluded (duplicates, reversed entries)."*

There's nothing to run in a cell for this step. Read CoCo's mapping and check a few labels against the table below.

<details>
<summary>Expected mapping (check against this)</summary>

**Department mapping:**

| Standard value | Raw labels |
|---|---|
| Front Desk | `Front Desk` |
| Food & Beverage | `F&B` |
| Housekeeping | `Housekeeping` |
| Management | `MGMT` |
| Spa & Wellness | `Spa & Wellness` |
| Operations | `Operations` |
| Facilities | `Facilities` |
| Marketing | `Marketing` |
| Admin | `Admin` |
| Finance | `Finance` |

**Category mapping (GL codes):**

| Standard value | GL code |
|---|---|
| Payroll | `PAY` |
| Utilities | `UTIL` |
| Maintenance | `MAINT` |
| Marketing | `MKTG` |
| Travel | `TRAVEL` |
| Office Supplies | `MISC` |
| Insurance | `INS` |
| Professional Services | `PROF` |

CoCo should also point out:

- **Dates:** three formats: `11/01/2026` (payroll), `Nov 01 2026` (GL), `2026-11-01` (bank).
- **Amounts:** text with dollar sign and comma (`"$4,200.00"`) in payroll, parenthesized debits (`(23,900.00)`) in GL, negative numbers (`-23900.00`) in bank.
- **Duplicate:** EMP-102 (James Rivera) on 11/15 with reference PAY-1007 appears twice in the payroll register.
- **Reversed:** JE-2014 has status `Reversed`. Its reference `TRV-1002-R` cancels JE-2013 (`TRV-1002`), so both should be excluded.
</details>

### STEP 3 — Build the unified TRANSACTIONS table

In the **same CoCo conversation**, send this prompt. CoCo reuses the mapping from STEP 2 and writes the SQL. Your SQL may not match the expected output word for word, and that's fine; the **check cell** below is what tells you it's right. Click **Allow** to run it.

> *"Using that mapping, create a Snowflake-managed Iceberg table MERIDIAN_STAY_FINANCE.CURATED.TRANSACTIONS (EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED') that combines the three raw tables into one transaction ledger:*
> - *Columns: transaction_id, transaction_date, description, amount, department, category, source_system, reference_id*
> - *Convert all dates to DATE type and all amounts to NUMBER(12,2) — make debits positive (absolute value)*
> - *Map department names and category codes to the standard values*
> - *Exclude reversed GL entries (status = 'Reversed') and the original entries they cancel (a reversal's reference_id is the original's reference_id plus '-R')*
> - *Remove the duplicate payroll row (same employee, same date, same reference_id — keep one)*
> - *For the bank statement, set category to 'Uncategorized' and department to 'Unknown' since those fields aren't in the export"*

Short on time? Paste the expected output into the empty cell below and run it.

<details>
<summary>Expected output (check against this)</summary>

```sql
CREATE OR REPLACE ICEBERG TABLE MERIDIAN_STAY_FINANCE.CURATED.TRANSACTIONS
  CATALOG = 'SNOWFLAKE'
  EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED'
AS
WITH payroll AS (
    SELECT
        reference_id AS transaction_id,
        TRY_TO_DATE(pay_date, 'MM/DD/YYYY') AS transaction_date,
        employee_name || ' - ' || pay_type AS description,
        TRY_TO_NUMBER(REPLACE(REPLACE(gross_amount, '$', ''), ',', ''), 12, 2) AS amount,
        CASE UPPER(TRIM(department))
            WHEN 'F&B' THEN 'Food & Beverage'
            WHEN 'MGMT' THEN 'Management'
            ELSE TRIM(department)
        END AS department,
        'Payroll' AS category,
        'Payroll Register' AS source_system,
        reference_id
    FROM MERIDIAN_STAY_FINANCE.RAW.PAYROLL_REGISTER
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY employee_id, pay_date, reference_id
        ORDER BY employee_id) = 1
),
gl AS (
    SELECT
        journal_id AS transaction_id,
        TRY_TO_DATE(posting_date, 'MON DD YYYY') AS transaction_date,
        description,
        ABS(TRY_TO_NUMBER(REPLACE(REPLACE(REPLACE(amount, '(', '-'), ')', ''), ',', ''), 12, 2)) AS amount,
        CASE UPPER(TRIM(department))
            WHEN 'MGMT' THEN 'Management'
            ELSE TRIM(department)
        END AS department,
        CASE UPPER(TRIM(category))
            WHEN 'PAY' THEN 'Payroll'
            WHEN 'UTIL' THEN 'Utilities'
            WHEN 'MAINT' THEN 'Maintenance'
            WHEN 'MKTG' THEN 'Marketing'
            WHEN 'TRAVEL' THEN 'Travel'
            WHEN 'MISC' THEN 'Office Supplies'
            WHEN 'INS' THEN 'Insurance'
            WHEN 'PROF' THEN 'Professional Services'
        END AS category,
        'General Ledger' AS source_system,
        reference_id
    FROM MERIDIAN_STAY_FINANCE.RAW.GL_JOURNAL_ENTRIES
    WHERE UPPER(TRIM(status)) != 'REVERSED'
      -- Also drop the original entry each reversal cancels (TRV-1002-R reverses TRV-1002)
      AND reference_id || '-R' NOT IN (
          SELECT reference_id FROM MERIDIAN_STAY_FINANCE.RAW.GL_JOURNAL_ENTRIES
          WHERE UPPER(TRIM(status)) = 'REVERSED'
      )
),
bank AS (
    SELECT
        transaction_id,
        TRY_TO_DATE(transaction_date, 'YYYY-MM-DD') AS transaction_date,
        description,
        ABS(TRY_TO_NUMBER(amount, 12, 2)) AS amount,
        'Unknown' AS department,
        'Uncategorized' AS category,
        'Bank Statement' AS source_system,
        transaction_id AS reference_id
    FROM MERIDIAN_STAY_FINANCE.RAW.BANK_STATEMENT
)
SELECT * FROM payroll
UNION ALL
SELECT * FROM gl
UNION ALL
SELECT * FROM bank;
```
</details>

### Check the result

Run this check. You should see **46 transactions**, **3 source systems**, **9 categories**, and **11 departments**.

That's 49 raw rows (13 payroll + 18 GL + 18 bank), minus 1 payroll duplicate and the 2 GL entries that cancel each other (JE-2013 and its reversal, JE-2014).

If your numbers are different, ask CoCo to fix it rather than editing the SQL yourself:
- More than 46 transactions: *"Which transactions appear more than once in MERIDIAN_STAY_FINANCE.CURATED.TRANSACTIONS? Fix the duplicate removal, and make sure both JE-2013 and its reversal JE-2014 are excluded."*
- Fewer than 46: *"Are all 12 unique payroll rows, 16 GL entries, and 18 bank rows included?"*

Short on time? Paste the expected output from STEP 3 into the empty cell above and run it.

In [ ]:
SELECT
    COUNT(*) AS transactions,
    COUNT(DISTINCT source_system) AS source_systems,
    COUNT(DISTINCT category) AS categories,
    COUNT(DISTINCT department) AS departments
FROM MERIDIAN_STAY_FINANCE.CURATED.TRANSACTIONS;

### Before and after

Now compare the rows that needed the most cleaning, as they arrived in `RAW` and as they are now in `CURATED`. Run it as-is.

- The **duplicate payroll row** (James Rivera, EMP-102, 11/15, PAY-1007) collapses into one.
- The **reversed GL entry** (JE-2014) and the **original it cancels** (JE-2013, $4,800 conference registration fees) are both gone.
- `F&B` → `Food & Beverage`, `MGMT` → `Management`, `PAY` → `Payroll`, `MAINT` → `Maintenance`.
- Dates and amounts now use one standard format.

In [ ]:
WITH raw_payroll AS (
    SELECT 'Before (RAW)' AS layer, 'PAYROLL' AS source, reference_id AS id,
           employee_name AS description, department, pay_date AS date_val, gross_amount AS amount_val
    FROM MERIDIAN_STAY_FINANCE.RAW.PAYROLL_REGISTER
    WHERE reference_id = 'PAY-1007'
),
raw_gl AS (
    SELECT 'Before (RAW)', 'GL', journal_id, description, department, posting_date, amount
    FROM MERIDIAN_STAY_FINANCE.RAW.GL_JOURNAL_ENTRIES
    WHERE journal_id IN ('JE-2013', 'JE-2014')
),
clean AS (
    SELECT 'After (CURATED)', source_system, transaction_id, description, department,
           transaction_date::STRING, amount::STRING
    FROM MERIDIAN_STAY_FINANCE.CURATED.TRANSACTIONS
    WHERE transaction_id IN ('PAY-1007', 'JE-2013', 'JE-2014')
)
SELECT * FROM raw_payroll
UNION ALL SELECT * FROM raw_gl
UNION ALL SELECT * FROM clean
ORDER BY id, layer DESC;

### STEP 4 — Find every reconciliation gap

A **reconciliation gap** is a transaction that appears in one system but has no matching entry in another. That's exactly what the auditors found last quarter with the $47,000 wire transfer.

The matching logic: a GL entry and a bank transaction **match** if they have the same absolute amount and their dates are within 3 days of each other.

> **One direction only:** This lab checks every bank payment against the GL. A full reconciliation also runs the other way, looking for GL entries with no bank payment yet, such as accruals or checks that haven't cleared.

Send this prompt to CoCo. Compare the output against the expected output below, then click **Allow** to run it.

> *"Create a Snowflake-managed Iceberg table MERIDIAN_STAY_FINANCE.CURATED.RECONCILIATION_GAPS (EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED') that identifies every bank transaction in CURATED.TRANSACTIONS with source_system = 'Bank Statement' that has no matching GL entry (source_system = 'General Ledger') with the same amount and a posting date within 3 days. Include:*
> - *Columns: gap_id, transaction_id, transaction_date, description, amount, source_system, gap_type, notes*
> - *Set gap_type to 'Bank Only - No GL Match'*
> - *Set notes to 'No corresponding general ledger entry found within 3 days'"*

Short on time? Paste the expected output into the empty cell below and run it.

<details>
<summary>Expected output (check against this)</summary>

```sql
CREATE OR REPLACE ICEBERG TABLE MERIDIAN_STAY_FINANCE.CURATED.RECONCILIATION_GAPS
  CATALOG = 'SNOWFLAKE'
  EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED'
AS
SELECT
    ROW_NUMBER() OVER (ORDER BY b.transaction_date, b.transaction_id) AS gap_id,
    b.transaction_id,
    b.transaction_date,
    b.description,
    b.amount,
    b.source_system,
    'Bank Only - No GL Match' AS gap_type,
    'No corresponding general ledger entry found within 3 days' AS notes
FROM MERIDIAN_STAY_FINANCE.CURATED.TRANSACTIONS b
LEFT JOIN MERIDIAN_STAY_FINANCE.CURATED.TRANSACTIONS g
  ON g.source_system = 'General Ledger'
  AND b.amount = g.amount
  AND ABS(DATEDIFF('day', b.transaction_date, g.transaction_date)) <= 3
WHERE b.source_system = 'Bank Statement'
  AND g.transaction_id IS NULL;
```
</details>

### Check the result

You should see **2 gaps** totaling **$47,200**:

| Gap | Transaction | Date | Amount | Description |
|---|---|---|---|---|
| 1 | BK-3006 | 2026-11-12 | $47,000.00 | WIRE TFR - GRAND BALLROOM VENUE DEPOSIT |
| 2 | BK-3018 | 2026-11-30 | $200.00 | MONTHLY ACCOUNT SERVICE FEE |

The first gap is the one leadership was afraid of: a **$47,000 venue deposit wire, paid on November 12 and never journaled**. It's the same kind of miss the auditors flagged last quarter, and it happened again. Someone authorized the payment, but nobody created the GL journal entry. But look at the second one: **BK-3018, $200, MONTHLY ACCOUNT SERVICE FEE**. This wasn't in anyone's close checklist. You just discovered it because you matched every bank transaction against the GL instead of spot-checking the big ones.

This is the kind of gap a manual close misses: too small to flag, too regular to question, and exactly the sort of thing that turns into an audit finding when someone finally looks.

Notice what's *not* in the ledger: the $4,800 conference registration fees (JE-2013) and their reversal (JE-2014). The registration was cancelled and never paid, so neither the charge nor its reversal belongs in November's books. Leave JE-2013 in and Travel spending is overstated by $4,800.

> **What about payroll?** Payroll is checked a different way. The payroll register lists individual paychecks, while the GL records each pay run as one batch. The register totals tie to the GL batches ($23,900 on Nov 1, $25,150 on Nov 15, $5,000 on Nov 30), but only because STEP 3 removed the duplicate row. Left in, James Rivera's second $3,800 paycheck would have put Nov 15 off by $3,800.

In [ ]:
SELECT COUNT(*) OVER () AS total_gaps,
       SUM(amount) OVER () AS total_gap_amount,
       *
FROM MERIDIAN_STAY_FINANCE.CURATED.RECONCILIATION_GAPS
ORDER BY amount DESC;

## See it: reconciliation at a glance

The clean `TRANSACTIONS` and `RECONCILIATION_GAPS` tables now power a shared **reconciliation dashboard**, a Streamlit app that's already in this repo.

1. In the Workspace file explorer, open **`reconciliation_dashboard/streamlit_app.py`**.
2. If a banner says *"This file looks like a Streamlit app, but is missing configuration"*, click **Convert to streamlit app**. The Workspace adds the configuration files the app needs.
3. Click **Run**. The app reads both `CURATED.TRANSACTIONS` and `CURATED.RECONCILIATION_GAPS`.

How to read it: **summary metrics** at the top show 16 of 18 bank transactions reconciled (89%), close readiness **Not Ready** (2 gaps remain), and $47,200 unreconciled. The **waterfall chart** traces November's cash flow from opening to closing balance — blue bars matched, red bars did not (with red triangles calling out the amounts). Below, **Investigate the gaps** shows one card per gap: what the bank paid, the GL entries around that date, and the root cause. **Next steps to close** lists the journal entries to post before the month can close.

## Data is ready → on to CoWork (in the UI)

You now have:

- `RAW.*` — the three exports, exactly as they arrived, in open Iceberg tables
- `CURATED.TRANSACTIONS` — 46 clean, standardized transactions in one vocabulary (Iceberg)
- `CURATED.RECONCILIATION_GAPS` — the 2 unmatched bank transactions totaling $47,200 (Iceberg)
- A live reconciliation dashboard anyone on the team can open

The last step happens in the Snowsight UI (no code): create a **Semantic View** over the two `CURATED` tables, create a **Cortex Agent** backed by it, and ask your questions in **Snowflake CoWork**. Follow the **"Ask It With CoWork"** section of the guide.

## Teardown

Once you're done with the lab, run the cell below to remove everything it created.

> **Note:** Dropping `MERIDIAN_STAY_FINANCE` cascades to everything inside it: Iceberg tables, stage, file format, semantic view, and Cortex Agent.

In [ ]:
-- TEARDOWN: remove all lab objects
USE ROLE ACCOUNTADMIN;

-- Drops all schemas, Iceberg tables, the stage, the semantic view, and the Cortex Agent
DROP DATABASE IF EXISTS MERIDIAN_STAY_FINANCE;

-- Removes the Git API integration created for the workspace (update the name if you used a different one)
DROP API INTEGRATION IF EXISTS GITHUB_MERIDIAN_FINANCE_LAB;